# 추가검증(K-fold 버전): 금융이력(씬/씩파일러) × 연령대 세그먼트별 C 영향력 비교

팀원의 LightGBM 코드(`C_effect_separate.py`)와 동일하게 **5-fold StratifiedKFold**로 OOF(out-of-fold) 예측을 만들어 전체 인구를 대상으로 평가.

**가설**: 금융이력이 없고(씬파일러) 나이가 적을수록, 사회연결망 데이터의 영향이 클 것이다.

**C군 두 버전 비교**: C_전체(15개) vs C_사회연결망만(3개)

캡 버전 하이퍼파라미터 그대로 사용 (n_estimators=200, max_depth=20, min_samples_leaf=20).

입력: `BASE_DIR = /content/drive/MyDrive/재계산` · 출력: `SAVE_DIR = /content/drive/MyDrive/26-3 부트캠프`

5번(단일 차원)·6번(교차 10개 셀) 세그먼트 표 + 8번에서 AUC 요약·전체인구 DeLong 요약·변수중요도·엑셀 결합 저장.

**9번: SHAP 방향성 분석** — 사회연결망 변수 3개가 부도확률에 +/- 어느 방향으로 작용하는지, 세그먼트별 평균 SHAP값 비교.

## 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from pandas.api.types import is_numeric_dtype

# 한글 폰트 설정 (Colab 환경)
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = "/content/drive/MyDrive/재계산"          # 데이터 입력 위치
SAVE_DIR = "/content/drive/MyDrive/26-3 부트캠프"    # 결과 저장 위치
os.makedirs(SAVE_DIR, exist_ok=True)

A_PATH = os.path.join(BASE_DIR, "model_A_financial_final.csv")
B_PATH = os.path.join(BASE_DIR, "model_B_nonfinancial_final.csv")
C_PATH = os.path.join(BASE_DIR, "model_C_nonfinancial_new_final.csv")

ID_COL = "SK_ID_CURR"
TARGET = "TARGET"
RANDOM_STATE = 42
N_FOLDS = 5

# 캡 버전 하이퍼파라미터 (rf_M1_M4.py에서 이미 진단 완료 — n_estimators=200)
RF_PARAMS = dict(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

## 데이터 로드 + 변수 정리

In [ ]:
df_a = pd.read_csv(A_PATH)
df_b = pd.read_csv(B_PATH)
df_c = pd.read_csv(C_PATH)

LOG_COLS = ["AMT_CREDIT_LOG", "AMT_ANNUITY_LOG", "AMT_GOODS_PRICE_LOG", "AMT_INCOME_TOTAL_LOG"]
df_a = df_a.drop(columns=[c for c in LOG_COLS if c in df_a.columns])

# B, C에도 TARGET이 중복으로 들어있어서, merge 시 컬럼명이 TARGET_x/TARGET_y로 안 꼬이게
# A만 TARGET을 남기고 B/C는 TARGET을 뺀 채로 merge
df = df_a.merge(df_b.drop(columns=[TARGET]), on=ID_COL).merge(df_c.drop(columns=[TARGET]), on=ID_COL)

# 연령대 세그먼트: AGE(소수형)를 10년 단위로 직접 구간화 (상한 np.inf로 열어둠)
AGE_BIN_EDGES = [20, 30, 40, 50, 60, np.inf]
AGE_BIN_LABELS = ["20대", "30대", "40대", "50대", "60대+"]
df["AGE_BAND"] = pd.cut(df["AGE"], bins=AGE_BIN_EDGES, labels=AGE_BIN_LABELS, right=False)

# bool -> int 변환, 숫자형이 아닌 컬럼(AGE_BAND 등)은 모델 입력에서 자동 제외
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

A_VARS = [c for c in df_a.columns if c not in (ID_COL, TARGET)]
B_VARS = [c for c in df_b.columns if c not in (ID_COL, TARGET)]
C_ALL_VARS = [c for c in df_c.columns if c not in (ID_COL, TARGET)]
C_SOCIAL_VARS = ["OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE", "SOCIAL_CIRCLE_MISSING_FLAG"]

non_numeric = [c for c in A_VARS + B_VARS + C_ALL_VARS if not is_numeric_dtype(df[c])]
A_VARS = [c for c in A_VARS if c not in non_numeric]
B_VARS = [c for c in B_VARS if c not in non_numeric]
C_ALL_VARS = [c for c in C_ALL_VARS if c not in non_numeric]
if non_numeric:
    print(f"모델 입력에서 자동 제외된 비수치형 컬럼: {non_numeric}")

M3_VARS = A_VARS + B_VARS
M4_ALL_VARS = A_VARS + B_VARS + C_ALL_VARS
M4_SOCIAL_VARS = A_VARS + B_VARS + C_SOCIAL_VARS

print(f"A_금융 {len(A_VARS)}개 / B_기존비금융 {len(B_VARS)}개 / "
      f"C_전체 {len(C_ALL_VARS)}개 / C_사회연결망 {len(C_SOCIAL_VARS)}개")
print(f"전체 {len(df):,}행, 부도율 {df[TARGET].mean():.2%}")

y = df[TARGET].values


def make_X(cols):
    return df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)

## K-fold OOF 예측 함수

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(df, y))


def get_oof(cols, name):
    X = make_X(cols)
    oof = np.zeros(len(y))
    imp_sum = np.zeros(len(cols))
    t0 = time.time()
    for i, (tr_idx, va_idx) in enumerate(folds, 1):
        clf = RandomForestClassifier(**RF_PARAMS)
        clf.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]
        imp_sum += clf.feature_importances_  # fold별 중요도 누적 (5개 fold 평균)
        print(f"  [{name}] fold {i}/{N_FOLDS} 완료 (누적 {time.time()-t0:.1f}초)")
    auc = roc_auc_score(y, oof)

    imp_avg = imp_sum / N_FOLDS
    imp_df = pd.DataFrame({"모델": name, "변수": cols, "중요도": imp_avg}).sort_values("중요도", ascending=False)
    imp_df["중요도_비율(%)"] = (imp_df["중요도"] / imp_df["중요도"].sum() * 100).round(2)

    print(f"[{name}] 전체(OOF) AUC={auc:.4f}  변수수={len(cols)}\n")
    return oof, auc, imp_df

## M3 / M4_전체C / M4_사회연결망 OOF 예측 생성

In [ ]:
oof_m3, auc_m3, imp_m3 = get_oof(M3_VARS, "M3_A_plus_B")
oof_m4_all, auc_m4_all, imp_m4_all = get_oof(M4_ALL_VARS, "M4_A_plus_B_plus_C전체")
oof_m4_social, auc_m4_social, imp_m4_social = get_oof(M4_SOCIAL_VARS, "M4_A_plus_B_plus_C사회연결망3개")

## DeLong test 함수

In [ ]:
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_T, m):
    n = preds_sorted_T.shape[1] - m
    pos = preds_sorted_T[:, :m]
    neg = preds_sorted_T[:, m:]
    k = preds_sorted_T.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_T[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_test(y_true, proba_1, proba_2):
    if len(np.unique(y_true)) < 2 or len(y_true) < 20:
        return {"AUC_1": np.nan, "AUC_2": np.nan, "AUC_차이": np.nan, "z": np.nan, "p_value": np.nan}
    order = np.argsort(-y_true)
    y_sorted = y_true[order]
    m = int(y_sorted.sum())
    preds = np.vstack([proba_1, proba_2])[:, order]
    aucs, cov = _fast_delong(preds, m)
    auc_diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = auc_diff / np.sqrt(var) if var > 0 else np.nan
    p = 2 * (1 - stats.norm.cdf(abs(z))) if not np.isnan(z) else np.nan
    return {"AUC_1": aucs[0], "AUC_2": aucs[1], "AUC_차이": auc_diff, "z": z, "p_value": p}


def eval_cell(mask, label):
    n_cell = mask.sum()
    if n_cell == 0:
        return None
    y_cell = y[mask]
    if len(np.unique(y_cell)) < 2:
        return None

    auc3 = roc_auc_score(y_cell, oof_m3[mask])
    auc4_all = roc_auc_score(y_cell, oof_m4_all[mask])
    auc4_social = roc_auc_score(y_cell, oof_m4_social[mask])
    dl_all = delong_test(y_cell, oof_m4_all[mask], oof_m3[mask])
    dl_social = delong_test(y_cell, oof_m4_social[mask], oof_m3[mask])

    return {
        "집단": label,
        "표본수": n_cell,
        "부도율(%)": round(y_cell.mean() * 100, 2),
        "M3_AUC": round(auc3, 4),
        "M4_전체C_AUC": round(auc4_all, 4),
        "M4_사회연결망_AUC": round(auc4_social, 4),
        "전체C_Δ": round(auc4_all - auc3, 4),
        "사회연결망_Δ": round(auc4_social - auc3, 4),
        "전체C_DeLong_p": dl_all["p_value"],
        "사회연결망_DeLong_p": dl_social["p_value"],
    }

## 세그먼트별 평가 (단일 차원 — 씬/씩, 연령대 각각)

In [ ]:
hist = df["BUREAU_NO_HISTORY_FLAG"].values
age_band = df["AGE_BAND"].astype(str).values

single_rows = []
single_rows.append(eval_cell(np.ones(len(y), dtype=bool), "전체"))
single_rows.append(eval_cell(hist == 1, "씬파일러(이력없음)"))
single_rows.append(eval_cell(hist == 0, "이력있음"))
for age_g in AGE_BIN_LABELS:
    single_rows.append(eval_cell(age_band == age_g, age_g))

single_df = pd.DataFrame([r for r in single_rows if r is not None])
print("=== [단일 차원] 씬파일러 여부 / 연령대별 결과 (K-fold OOF, 전체 인구 기준) ===")
print(single_df.to_string(index=False))

## 세그먼트별 평가 (교차 — 연령대 × 씬/씩 10개 셀)

In [ ]:
cross_rows = []
for age_g in AGE_BIN_LABELS:
    for hist_v, hist_label in {0: "이력있음", 1: "씬파일러"}.items():
        mask = (age_band == age_g) & (hist == hist_v)
        row = eval_cell(mask, f"{age_g}·{hist_label}")
        if row is not None:
            cross_rows.append(row)

cross_df = pd.DataFrame(cross_rows)
print("\n=== [교차] 연령대 × 금융이력 10개 셀 결과 (K-fold OOF, 전체 인구 기준) ===")
print(cross_df.to_string(index=False))

## 시각화: 교차 셀별 AUC 상승폭(Δ) 비교

In [ ]:
cross_df["연령대"] = cross_df["집단"].str.split("·").str[0]
cross_df["금융이력"] = cross_df["집단"].str.split("·").str[1]

pivot_all = cross_df.pivot(index="연령대", columns="금융이력", values="전체C_Δ").reindex(AGE_BIN_LABELS)
pivot_social = cross_df.pivot(index="연령대", columns="금융이력", values="사회연결망_Δ").reindex(AGE_BIN_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
pivot_all.plot(kind="bar", ax=axes[0], color=["#1E2761", "#F2A93B"])
axes[0].set_title("C 전체 추가 시 AUC 상승폭(Δ) — K-fold OOF 기준")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("AUC 상승폭(M4-M3)")

pivot_social.plot(kind="bar", ax=axes[1], color=["#1E2761", "#F2A93B"])
axes[1].set_title("사회연결망 3개만 추가 시 AUC 상승폭(Δ)")
axes[1].axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "rf_kfold_segment_delta.png"), dpi=150)
plt.show()

## 결과 저장

In [ ]:
single_df.to_csv(os.path.join(SAVE_DIR, "rf_kfold_segment_single.csv"), index=False, encoding="utf-8-sig")
cross_df.to_csv(os.path.join(SAVE_DIR, "rf_kfold_segment_cross.csv"), index=False, encoding="utf-8-sig")

# single + cross를 한 엑셀 파일에 위/아래로 이어붙이기 (컬럼이 같아서 그대로 세로로 결합)
single_out = single_df.copy()
single_out.insert(0, "구분", "단일차원")

cross_out = cross_df.copy()
cross_out.insert(0, "구분", "교차")

combined_df = pd.concat([single_out, cross_out], ignore_index=True)
EXCEL_PATH = os.path.join(SAVE_DIR, "rf_kfold_segment_결과.xlsx")
combined_df.to_excel(EXCEL_PATH, index=False)
print(f"엑셀 저장 완료: {EXCEL_PATH}")

# 변수별 A/B/C 군 표시
def tag_group(v):
    if v in A_VARS:
        return "A_금융"
    if v in B_VARS:
        return "B_기존비금융"
    if v in C_ALL_VARS:
        return "C_신규비금융"
    return "기타"


imp_m3["군"] = imp_m3["변수"].apply(tag_group)
imp_m4_all["군"] = imp_m4_all["변수"].apply(tag_group)
imp_m4_social["군"] = imp_m4_social["변수"].apply(tag_group)

# AUC 요약
auc_summary = pd.DataFrame([
    {"모델": "M3 (A+B)", "변수수": len(M3_VARS), "AUC": round(auc_m3, 4)},
    {"모델": "M4_전체C (A+B+C15개)", "변수수": len(M4_ALL_VARS), "AUC": round(auc_m4_all, 4)},
    {"모델": "M4_사회연결망 (A+B+C3개)", "변수수": len(M4_SOCIAL_VARS), "AUC": round(auc_m4_social, 4)},
])
print("=== AUC 요약 (K-fold OOF 기준) ===")
print(auc_summary.to_string(index=False))

# 전체 인구 기준 DeLong test (세그먼트로 나누기 전, M3 대비 전체 유의성)
dl_all = delong_test(y, oof_m4_all, oof_m3)
dl_social = delong_test(y, oof_m4_social, oof_m3)

delong_summary = pd.DataFrame([
    {"비교": "M4_전체C vs M3", **dl_all},
    {"비교": "M4_사회연결망 vs M3", **dl_social},
])
print("\n=== DeLong test 요약 (전체 인구, K-fold OOF 기준) ===")
print(delong_summary.to_string(index=False))
delong_summary.to_csv(os.path.join(SAVE_DIR, "rf_kfold_delong_summary.csv"), index=False, encoding="utf-8-sig")

auc_summary.to_csv(os.path.join(SAVE_DIR, "rf_kfold_auc_summary.csv"), index=False, encoding="utf-8-sig")
imp_m3.to_csv(os.path.join(SAVE_DIR, "rf_kfold_M3_feature_importance.csv"), index=False, encoding="utf-8-sig")
imp_m4_all.to_csv(os.path.join(SAVE_DIR, "rf_kfold_M4_전체C_feature_importance.csv"), index=False, encoding="utf-8-sig")
imp_m4_social.to_csv(os.path.join(SAVE_DIR, "rf_kfold_M4_사회연결망_feature_importance.csv"), index=False, encoding="utf-8-sig")

print(f"\n결과 저장 완료: {SAVE_DIR}")

print("\n※ 가설 검증 포인트: '20대 × 씬파일러' 셀의 전체C_Δ·사회연결망_Δ가 다른 셀보다 "
      "크고 DeLong p<0.05인지 확인. 팀원 LightGBM 결과(K-fold)와 함께 비교할 것.")

## SHAP 방향성 분석 (사회연결망 변수 3개 중심)

In [ ]:
import shap

# SHAP은 평가용이 아니라 해석용이라, 전체 데이터로 한 번 더 학습한 모델을 사용
clf_social_full = RandomForestClassifier(**RF_PARAMS)
clf_social_full.fit(make_X(M4_SOCIAL_VARS), y)

explainer = shap.TreeExplainer(clf_social_full)

# 전체 30만행은 오래 걸려서 5,000명 샘플로 계산
SAMPLE_N = 5000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(df), size=SAMPLE_N, replace=False)
X_sample = make_X(M4_SOCIAL_VARS).iloc[sample_idx]

shap_values = explainer.shap_values(X_sample)
# shap 버전에 따라 반환 형태가 달라서 방어적으로 처리 (TARGET=1, 부도 기준만 사용)
if isinstance(shap_values, list):
    shap_pos = shap_values[1]
elif shap_values.ndim == 3:
    shap_pos = shap_values[:, :, 1]
else:
    shap_pos = shap_values

# (1) 전체 방향성 요약 플롯 — 사회연결망 3개 변수가 부도확률에 +인지 -인지 한눈에 확인
shap.summary_plot(shap_pos, X_sample, show=False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_social_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

# (2) 세그먼트(연령대×이력)별 평균 SHAP값 비교 — 방향·크기가 세그먼트마다 다른지 확인
sample_age = age_band[sample_idx]
sample_hist = hist[sample_idx]

shap_seg_rows = []
for age_g in AGE_BIN_LABELS:
    for hist_v, hist_label in {0: "이력있음", 1: "씬파일러"}.items():
        m = (sample_age == age_g) & (sample_hist == hist_v)
        if m.sum() < 20:
            continue
        row = {"연령대": age_g, "금융이력": hist_label, "표본수": int(m.sum())}
        for j, col in enumerate(M4_SOCIAL_VARS):
            row[f"{col}_평균SHAP"] = round(float(shap_pos[m, j].mean()), 5)
        shap_seg_rows.append(row)

shap_seg_df = pd.DataFrame(shap_seg_rows)
print(shap_seg_df.to_string(index=False))
shap_seg_df.to_csv(os.path.join(SAVE_DIR, "rf_kfold_shap_segment.csv"), index=False, encoding="utf-8-sig")